In [1]:
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA

from model_ranking import (
    get_precomputed_feature_path,
    reshape_features_for_analysis,
    load_h5,
)

INFO: P [MainThread] 2025-08-22 09:26:05,025 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/predictor.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
path = get_precomputed_feature_path(
    model_name="E_model5",
    target="EPFL",
    base_path="/scratch/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled"
)

In [3]:
layer_key = "decoders.2"

features = load_h5(path, f"{layer_key}_features")
labels = load_h5(path, f"{layer_key}_labels")
print(f"Loaded features shape: {features.shape}")
print(f"Loaded labels shape: {labels.shape}")

Loaded features shape: (750, 1000, 32)
Loaded labels shape: (750, 1000)


In [4]:
features_reshaped, labels_reshaped = reshape_features_for_analysis(features, labels)
print(f"reshaped features shape {features_reshaped.shape}")
print(f"reshaped labels shape {labels_reshaped.shape}")

Analyzing all 750 patches with 750000 total pixels
reshaped features shape (750000, 32)
reshaped labels shape (750000,)


In [5]:
features_pca = PCA(  # pyright: ignore[reportUnknownVariableType]
        n_components=0.8, random_state=42
    ).fit_transform(features_reshaped)

In [6]:
features_pca.shape

(750000, 1)

In [7]:
def gmm_bic_score(estimator, X):
    """Callable to pass to GridSearchCV that will use the BIC score."""
    # Make it negative since GridSearchCV expects a score to maximize
    return -estimator.bic(X)


param_grid = {
    "n_components": range(1, 10),
    "covariance_type": ["full"],
}
grid_search = GridSearchCV(
    GaussianMixture(), param_grid=param_grid, scoring=gmm_bic_score
)
grid_search.fit(features_pca)

/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/sklearn/mixture/_base.py:275: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/sklearn/mixture/_base.py:275: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/sklearn/mixture/_base.py:275: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/sklearn/mixture/_base.py:275: ConvergenceWarning: Best performing in

,estimator,GaussianMixture()
,param_grid,"{'covariance_type': ['full'], 'n_components': range(1, 10)}"
,scoring,<function gmm...x7ffee3d8ec00>
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_components,7


In [1]:
import pandas as pd

df = pd.DataFrame(grid_search.cv_results_)[
    ["param_n_components", "param_covariance_type", "mean_test_score"]
]
df["mean_test_score"] = -df["mean_test_score"]
df = df.rename(
    columns={
        "param_n_components": "Number of components",
        "param_covariance_type": "Type of covariance",
        "mean_test_score": "BIC score",
    }
)
df.sort_values(by="BIC score")

NameError: name 'grid_search' is not defined